# Notebook 99 -- human_gold_arm

## Read this before running anything

The preregistration is explicit. Prereg 6.7 has an ensemble of language models
code the open-ends **in place of human raters**, and it prescribes **no
human-coding step** anywhere. Everything in this notebook -- the coder materials,
the inter-rater reliability, the adjudicated consensus, and gate clause A
(agreement between the ensemble and human coders) -- is an **addition** to the
registration, not an implementation of it.

That makes this arm a **DEVIATION**, and it requires a **dated amendment to the
OSF registration, posted BEFORE any coding happens**. The amendment carries the
codebook, the coder instructions, the reliability statistics, the human-match
thresholds, and the frozen SEED/TEST split. Coding that begins before the
amendment is posted cannot be reported as planned analysis, and no printout in
this notebook can repair that afterwards.

**Asks:** OPTIONAL -- the human-coder arm. Only for gate clause A.

**Reads:** `strata_sample.csv` and `gold_split_assignment.csv` (notebook 03),
`ensemble_scores.csv` (notebook 07, optional), and the coder files the coders
return.

**Writes:** `codebook_bottom_up.txt`, `gold_codes.csv`, `gold_irr_by_facet.csv`,
`human_match_results.csv`.

## Nothing in the default chain depends on this notebook

The registered gate has two clauses and both run with no human anywhere near
them: **(B)** the models agree at an intraclass correlation (ICC, a measure of
how much independent raters scoring the same things agree) of at least .60, and
**(C)** each facet correlates at least .30 with its matching survey scale. Those
two are computed elsewhere and are unaffected by whether this notebook is ever
run. Clause A is ours, it only **adds** to the gate, and it never relaxes it.

Where clause A does not run, it is reported as **"not run"** -- never as a blank
and never as a NaN, either of which a later reader could mistake for a facet that
was tested against humans and failed.

## The HUMAN_GOLD switch decides whether anyone reads the output

`HUMAN_GOLD` in the config cell is what tells the rest of the chain that a human
arm exists. Notebooks 04, 10 and 11 ask for this notebook's files **only when it
is True**, and notebook 12 reports the human arm as NOT RUN when it is False. So
a run with the switch off writes all four files correctly and no other notebook
opens any of them. The setup cell says so on its own line, loudly, because the
manifest will also record `human_gold=False` against a run that just executed the
whole deviation arm.

## Who reads what this writes

`gold_codes.csv` goes to notebook 04 (human-coded reference passages) and
notebook 11 (a non-model corroborator for E4). `human_match_results.csv` goes to
notebook 10, which reads its `scope == "pooled"` rows as gate clause A and its
shared-error flag as the cap on clause B. `gold_irr_by_facet.csv` is read by no
notebook at all: it exists for the OSF amendment, and the reading it carries has
to reach the write-up by hand.

## One row per participant per item, checked here

Every file in this chain is keyed on `response_id` x `item_type` -- 948
participants x 3 open-ended items. Nothing upstream guarantees that key is
unique, so this notebook checks it on every file it reads on those keys and on
every file it writes on them. A duplicated row does not announce itself: it
inflates a sample size while the printed count stays right.

## It runs in two passes

**Pass A** -- with no coder files in the folder, this notebook writes the
bottom-up codebook, prints what to send out and where the returned files go, and
finishes. Nothing errors and nothing half-writes.

**Pass B** -- with two or more usable coder files in the folder, it computes
inter-rater reliability, adjudicates the coders into one consensus label per row,
and (if the ensemble has been scored) computes clause A on the TEST half only.

## Config

In [ ]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "99"                     # this notebook's number
NB_NAME     = "human_gold_arm"         # this notebook's job, in two words
NB_ASKS     = "OPTIONAL: the human-coder arm. Only for gate clause A."

## Setup

In [ ]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

In [ ]:
# === SETUP (cont.): the two packages a Colab machine does not ship ===
# WHAT THIS CELL DOES: installs the two agreement packages this notebook needs,
# then records the versions the session actually resolved. It computes nothing.
#
# WHY IT RUNS EVERY TIME: a Colab machine is thrown away when the runtime
# disconnects, so the install happens at the top of every session.
#
#   pingouin      gives us the intraclass correlation (ICC) -- how much of the
#                 variation in a set of scores is real differences between
#                 responses rather than differences between raters. Clause A
#                 uses it wherever both the ensemble and the humans scored a
#                 facet as a graded quantity rather than present/absent.
#   krippendorff  gives us Krippendorff's alpha, an agreement coefficient built
#                 for LABELS rather than numbers, which is what the holistic
#                 principled/vengeful/both/neither call is.
#
# WHY THE COMPILED CORE IS NEVER PINNED HERE. numpy, pandas and scipy ship in
# the Colab image compiled against one another. A pip install that resolves a
# different numpy leaves the already-built wheels pointing at the wrong C
# structures, and the next import dies with
#     ValueError: numpy.dtype size changed, may indicate binary incompatibility
# Nothing in this notebook needs a different numpy, so nothing asks for one.
# IF YOU EVER SEE THAT ERROR: pip cannot swap a compiled module the live kernel
# has already imported. Runtime -> Restart session, then run from the top.
!pip install -q --no-deps krippendorff
!pip install -q pingouin

# NO "installed" LINE HERE. A cell cannot see pip's exit status, so a printed
# claim of success would appear just as readily after a failed install and would
# contradict the block below. The version block is the one that tells the truth: a
# package that did not install reports NOT AVAILABLE against its name.
#
# Reproducibility record: the resolved versions belong in the OSF bundle next to
# the results, so a later run can be compared against this one.
import importlib, sys
from importlib import metadata as _md

def _ver(mod_name, dist_name=None):
    # Not every package exposes __version__ (krippendorff does not), so fall back
    # to the installed distribution metadata before giving up.
    try:
        return importlib.import_module(mod_name).__version__
    except Exception:
        pass
    try:
        return _md.version(dist_name or mod_name)
    except Exception as e:
        return f"NOT AVAILABLE -- {type(e).__name__}"

print("Resolved versions (archive these with the outputs):")
print(f"  python        {sys.version.split()[0]}")
for _mod in ["numpy", "pandas", "krippendorff", "pingouin"]:
    print(f"  {_mod:13s} " + str(_ver(_mod)))

In [ ]:
# === SETUP (cont.): imports and the canonical facet vocabulary ===
import pandas as pd
import numpy as np
import glob
from collections import Counter
import warnings

# NOTHING IS SILENCED GLOBALLY HERE, and the omission is deliberate. A blanket
# warnings.filterwarnings("ignore") stays in force for the rest of the Colab
# session, so it also silences whatever a reader runs after this notebook. It
# would take with it the warnings that carry information: scipy's
# near-constant-input notice, which is what tells a reader that a facet nobody
# varied is being correlated with something, and any ill-conditioning note from a
# model fit. Narrowing to RuntimeWarning would not save those two either --
# scipy's ConstantInputWarning and NearConstantInputWarning are both subclasses
# of RuntimeWarning, so the narrow filter swallows them as readily as the wide
# one. The one place a flood is plausible is the bootstrap loop, where a
# degenerate draw can raise thousands of numpy divide-by-zero notices; that loop
# suppresses RuntimeWarning around itself and nowhere else. See DO (14, cont.).

import krippendorff          # nominal alpha on the holistic call
import pingouin as pg        # ICC for the graded facets

# The reliability tables are wide. Without these two settings pandas replaces the
# middle columns with an ellipsis, and a truncated agreement table is worse than
# no agreement table.
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

# --- The key every file in this chain is keyed on --------------------------
# One row per participant per open-ended item. Every merge, every alignment and
# every uniqueness check below uses this pair and nothing else. response_id alone
# would be wrong: one participant wrote up to three texts and carries up to three
# different desert scores.
KEY = ["response_id", "item_type"]

# --- The canonical facet vocabulary (prereg 5.15; design section 2) ---------
# Nine multi-label facets. A coder scores each 0 (absent) or 1 (clearly present),
# or 0-2 for intensity; several may be 1 on one response and none is ever forced
# to win. Co-occurrence is the object of measurement for E4, so a scheme that made
# the coder pick one winner would destroy the quantity the study is after.
# The order is fixed and matches every other file in the chain.
FACET_NAMES = ["desert", "proportionality", "censure", "moral_balance",
               "revenge", "suffering", "deterrence", "incapacitation",
               "rehabilitation"]

# The holistic principled/vengeful/both/neither call is collected but kept OUT of
# the nine-facet vector. It comes from the same reader in the same pass, so it is
# descriptive coverage and a reliability target only -- never corroboration for
# facet-level co-occurrence, which would be one reading counted twice
# (design section 9, rule 6).
HOLISTIC_LEVELS = ["principled", "vengeful", "both", "neither"]

print(f"{len(FACET_NAMES)} facets, keyed on {KEY[0]} x {KEY[1]}.")

In [ ]:
# === SETUP (cont.): the switch that decides whether anyone reads this ===
# HUMAN_GOLD is deliberately NOT branched on below. A reader who opened this
# notebook meant to run the human arm, and refusing to run it would waste a coding
# pass that has already been paid for. What the switch decides is whether the REST
# of the chain will open what this notebook writes:
#
#   notebook 04  asks for gold_codes.csv only when HUMAN_GOLD is True
#   notebook 10  asks for human_match_results.csv only when HUMAN_GOLD is True,
#                and records gate clause A as "not run" for every facet otherwise
#   notebook 11  asks for gold_codes.csv only when HUMAN_GOLD is True
#   notebook 12  reports the human arm as NOT RUN in the OSF bundle manifest
#
# and done() in the setup cell stamps human_gold=False into _manifest.json against
# this run. So a run with the switch off produces four correct files that nothing
# opens, beside a manifest saying the arm did not run. That contradiction is worth
# one loud paragraph rather than a silent inconsistency a reader finds in March.
if not HUMAN_GOLD:
    print("=" * 72)
    print("HUMAN_GOLD IS FALSE, AND THIS NOTEBOOK IS THE HUMAN ARM.")
    print("  Everything below still runs and every file is still written. But")
    print("  notebooks 04, 10 and 11 will not open any of them, notebook 12 will")
    print("  report the human arm as NOT RUN, and _manifest.json will record")
    print("  human_gold=False against this run.")
    print("  TO USE WHAT THIS NOTEBOOK PRODUCES: set HUMAN_GOLD = True in the")
    print("  config cell above, run this notebook again, then re-run 04, 10, 11")
    print("  and 12. The dated OSF amendment has to be posted either way, and")
    print("  before the coding, not after it.")
    print("=" * 72)
else:
    print("HUMAN_GOLD is True: notebooks 04, 10 and 11 will read this arm's files.")

In [ ]:
# === SETUP (cont.): the five numbers this notebook decides on ===
HUMAN_IRR_MIN     = 0.60   # human-side reliability. Mirrors the registered
                           # ICC >= .60, applied to the coders instead of the
                           # models: if humans cannot agree on a facet, no
                           # automated match can rescue it (design section 8).
HUMAN_IRR_MIN_POS = 10     # and the floor below which an agreement coefficient
                           # between the coders is not read as reliability at all.
                           # Two coders who both marked a facet absent on every
                           # row agree perfectly: Gwet's AC1 returns 1.00 with a
                           # standard error of 0.00, and the facet nobody could
                           # see would read as the best-measured facet in the
                           # table. Design section 7 predicts exactly this for
                           # proportionality at its ~1.0% corpus prevalence, in
                           # those words -- "AC1 dominated by the all-zero cell".
                           # 10 positives is a bare floor: at about 14 the 95%
                           # half-width already runs to +/-0.26. OPEN: design
                           # section 11 item 3 leaves the minimum positive-case
                           # count for the team to ratify.
CLAUSE_A_MIN      = 0.60   # clause A, read on the LOWER BOUND of the bootstrap
                           # 95% confidence interval, not the point estimate.
CLAUSE_A_MIN_POS  = 30     # and a minimum count of human-coded POSITIVE cases,
                           # below which the interval is too wide to interpret
                           # wherever it happens to sit (design section 7).
ENS_PRESENT       = 0.50   # what counts as "the ensemble saw this facet". The
                           # models score each facet 0.0 to 1.0 and the ensemble
                           # mean averages them, so the continuous score needs a
                           # rule before it can be set beside a human 0/1 code.
                           # Above .50 means a majority of the models called the
                           # facet present, which is the same threshold notebook
                           # 10 uses wherever a facet has to be COUNTED rather
                           # than correlated. A threshold of zero would instead
                           # count the facet present wherever any single model
                           # returned 0.001 -- that is nearly every row, the
                           # ensemble column becomes a constant, and the
                           # coefficient then returns the same number for a
                           # perfect ensemble, an uninformative one and an
                           # exactly inverted one. OPEN: design section 11 item
                           # 10 leaves the presence threshold for the team to
                           # ratify, and clause A's estimate moves with it, so
                           # any number quoted from this notebook has to name the
                           # threshold it was computed at.

print(f"Human-side reliability bar AC1 >= {HUMAN_IRR_MIN:.2f} on at least "
      f"{HUMAN_IRR_MIN_POS} positive cases;")
print(f"clause A needs CI lower bound >= {CLAUSE_A_MIN:.2f} and n_pos >= "
      f"{CLAUSE_A_MIN_POS}; ensemble presence threshold {ENS_PRESENT:.2f}.")

## Load

In [ ]:
# === LOAD (1 of 3): the drawn sample and the frozen split ===
# WHAT THIS CELL DOES: fetches the two files notebook 03 wrote, plus the ensemble
# scores if they have been produced yet, and checks both required files against
# their column contract and against the one-row-per-key contract.
# CONSUMES:
#   strata_sample.csv         (03) -- the roughly 250-300 responses drawn on three
#     model-free routes, already carrying blank facet columns. This file IS the
#     coder template: it is what goes out and what comes back filled in. Only the
#     id pair, the verbatim text and the stratum travel with it -- no survey
#     scores, no word-list counts, no model output -- because a coder blind to all
#     of that is the whole basis for treating human codes as an independent
#     criterion (design section 8).
#   gold_split_assignment.csv (03) -- which of those rows are SEED and which TEST,
#     frozen from RANDOM_SEED before any scoring so it cannot be tuned once the
#     results are visible.
#   ensemble_scores.csv       (07) -- OPTIONAL here. Clause A compares the ensemble
#     against the humans, so it needs this file; the coding, the reliability and
#     the adjudication below do not. Absent, this notebook still produces the
#     consensus gold codes and reports clause A as "not run".
print("Inputs:")
P_SAMPLE = need("strata_sample.csv",         produced_by="03")
P_SPLIT  = need("gold_split_assignment.csv", produced_by="03")
P_ENS    = need("ensemble_scores.csv",       produced_by="07", optional=True)

# keep_default_na=False, matching how notebook 03 wrote the file: a participant who
# literally answered "N/A" keeps that text instead of arriving as a missing value.
# These texts are what a coder reads, so a real answer read as missing would be
# sent out as a blank row and waste a whole coding pass.
template = pd.read_csv(P_SAMPLE, keep_default_na=False)
split    = pd.read_csv(P_SPLIT,  keep_default_na=False)
template["text"] = template["text"].astype(str)

# --- The column contract, on BOTH required files ---------------------------
# issubset rather than equality, so a file carrying extra columns still passes.
# Checking the split file here as well as the sample means a bad split names
# itself instead of surfacing as a raw KeyError on split["gold_split"] two lines
# down, or on the merge in DO (12) forty cells later.
assert {"response_id", "item_type", "text", "stratum"}.issubset(template.columns), \
    ("strata_sample.csv is missing one of response_id / item_type / text / "
     f"stratum; it carries {sorted(template.columns)[:12]}. Re-run notebook 03.")
assert {"response_id", "item_type", "gold_split"}.issubset(split.columns), \
    ("gold_split_assignment.csv is missing one of response_id / item_type / "
     f"gold_split; it carries {sorted(split.columns)}. Re-run notebook 03.")

# --- The one-row-per-(response_id, item_type) contract on the template -----
# This one is checked here because it is USED here rather than merged: DO (12)
# looks the text and the stratum up with template[...].iloc[0], which takes the
# first match and says nothing about the second. A duplicated template row would
# also have gone out to the coders twice. The same contract on the split file is
# checked immediately before the merge that would fan out on it, in DO (12).
assert not template.duplicated(KEY).any(), (
    "strata_sample.csv holds more than one row for some response_id x item_type: "
    + str(template[template.duplicated(KEY, keep=False)][KEY]
          .head(4).to_dict("records"))
    + ". The sample is one row per participant per item. Re-run notebook 03.")

print(f"  drawn sample: {len(template)} rows across "
      f"{template['stratum'].nunique()} strata, one row per {KEY[0]} x {KEY[1]}.")
print(f"  frozen split: {split['gold_split'].value_counts().to_dict()}")

In [ ]:
# === LOAD (2 of 3): the returned coder files ===
# WHAT THIS CELL DOES: looks in the working folder for the spreadsheets the coders
# sent back, and decides whether there is anything to adjudicate.
#
# HOW THE FILES GET HERE. There is no upload dialog anywhere in this chain except
# the raw export in notebook 00. Put the returned files in the working folder
# printed above -- in Colab, open the file browser on the left, find that folder,
# and drag them in -- then run this cell. Three name patterns are accepted, so a
# coder can return coder_A.csv, A_codes.csv or gold_codes_A.csv and be recognised
# either way.
#
# HAZARD: this notebook WRITES gold_codes.csv into the same folder. Left in the
# search, its own consensus output would be picked up on the next run as one more
# coder -- a consensus agreeing with itself -- and would inflate every number
# below. The notebook's own outputs are excluded by name.
OUR_OUTPUTS = {"gold_codes.csv", "gold_irr_by_facet.csv", "human_match_results.csv"}
_found = (set(glob.glob("gold_codes*.csv")) | set(glob.glob("*_codes.csv"))
          | set(glob.glob("coder_*.csv")))
coder_files = sorted(f for f in _found if f not in OUR_OUTPUTS)

# Two coders is the minimum that makes reliability a question at all, so it is also
# the trigger for the second pass. With fewer, every cell below prints what it
# would have done and the notebook finishes cleanly -- no error, no half-written
# file, nothing for a later reader to mistake for a result. This counts FILES; DO
# (6) counts usable RATERS and can turn the switch back off.
HAVE_CODERS = len(coder_files) >= 2
print(f"Coder files found: {coder_files if coder_files else 'none'}")

# Bind now, so a run with no coder files walks the whole notebook and reports
# rather than dying on an undefined name halfway down. CLAUSE_A_STATUS carries the
# string "not run", never False: "this was never computed" and "this was tested
# against humans and failed" are different findings, and a False here would
# manufacture the second one out of the first.
coders, cids, common_idx = {}, [], None
irr_df, gold_codes = pd.DataFrame(), None

# Clause A's rows are collected in THREE lists, one per scope, and each list is
# emptied at the top of the cell that fills it. One shared list appended to by four
# cells cannot be re-run: changing CLAUSE_A_MIN and re-running the clause A block
# would append a SECOND set of pooled rows, notebook 10 would find two pooled rows
# per facet and record clause A as "not run" for every facet, and the only sign
# would be a row count in DO (17) that a reader had to notice was double.
hm_pooled, hm_stratum, hm_class_rows = [], [], []
CLAUSE_A_STATUS = "not run"
NOT_RUN_REASON  = ("fewer than two coder files in the working folder"
                   if not HAVE_CODERS else "")

In [ ]:
# === LOAD (3 of 3): the ensemble scores, one column per facet ===
# WHAT THIS CELL DOES: reads the long ensemble file and turns it wide, so the
# clause A code below can ask for one named column per facet.
#
# WHY THE RESHAPE. Notebook 07 exports long -- one row per response x item x facet
# -- because every downstream analysis groups by facet. Clause A does the
# opposite: it walks the facets one at a time and needs the ensemble's number, and
# each model's number, sitting on the same row as the human code, so the two
# frames can be joined on response_id x item_type.
ENSEMBLE_AVAILABLE = P_ENS is not None
ens_wide, MODELS = None, []
if ENSEMBLE_AVAILABLE:
    ens_long = pd.read_csv(P_ENS)
    # The rater roster is read off the columns rather than hardcoded, so the number
    # of models reported is always the number of models there actually were.
    MODELS = [c[len("score_"):] for c in ens_long.columns if c.startswith("score_")]
    # The joint nine-facet call only. Notebook 07 may also carry desert_solo /
    # revenge_solo rows from the separate elicitations; those belong to the E4 halo
    # diagnostic, and folding them in here would compare the humans against a
    # different instrument from the one clause A is about.
    ens_long = ens_long[ens_long["facet"].isin(FACET_NAMES)].copy()

    # THE ONE-ROW-PER-KEY CONTRACT ON THIS FILE, REPORTED AND NOT RAISED.
    # ensemble_scores.csv is an OPTIONAL input here: a defect in it costs clause A
    # and must not stop the coding, the reliability or the adjudication, none of
    # which read it. The pivot below would fail on a duplicated key with a bare
    # "Index contains duplicate entries, cannot reshape", so the check is made
    # first and the problem is named in this notebook's own words.
    _dup = int(ens_long.duplicated(KEY + ["facet"]).sum())
    if _dup or not len(ens_long):
        ENSEMBLE_AVAILABLE, MODELS = False, []
        NOT_RUN_REASON = NOT_RUN_REASON or (
            f"ensemble_scores.csv holds {_dup} duplicated response_id x item_type "
            "x facet rows" if _dup else
            "ensemble_scores.csv carries none of the nine joint-call facets")
        print(f"  ensemble_scores.csv cannot be used: {NOT_RUN_REASON}.")
        print("  Reported rather than raised: this file is optional here. Re-run")
        print("  notebook 07, then re-run this notebook for clause A.")
    else:
        # pivot with index=KEY is what GUARANTEES ens_wide is one row per
        # response_id x item_type: pandas raises rather than fan out, and the check
        # above has already ruled that out. DO (15) re-states the guarantee as an
        # assertion at the merge, because the two cells can be run apart.
        ens_wide = ens_long.pivot(index=KEY, columns="facet",
                                  values="ensemble_mean")
        ens_wide.columns = [f"{c}_ens_mean" for c in ens_wide.columns]
        for m in MODELS:
            w = ens_long.pivot(index=KEY, columns="facet", values=f"score_{m}")
            w.columns = [f"{c}__{m}" for c in w.columns]
            ens_wide = ens_wide.join(w)
        ens_wide = ens_wide.reset_index()
        print(f"  ensemble scores: {len(ens_wide)} response x item rows, "
              f"{len(MODELS)} rater(s): {', '.join(MODELS)}")

if not ENSEMBLE_AVAILABLE:
    NOT_RUN_REASON = NOT_RUN_REASON or "ensemble_scores.csv is not in the folder"
    print("The coding, the reliability and the adjudicated consensus below still")
    print("run. Clause A (ensemble against humans) reports 'not run'.")

## Do

In [ ]:
# === DO (1 of 17): the coder materials come in TWO variants ===
# WHY TWO (design section 8). Handing humans the same pinned token decisions and
# the same anchor examples the language-model rubric uses imports the shared blind
# spot straight into the "independent" anchor: if a boundary call in the codebook
# is itself wrong -- treating "pay for what he did" as pure desert when many
# writers mean payback -- then humans and models agree BECAUSE THEY WERE INSTRUCTED
# IDENTICALLY, and that agreement gets reported as accuracy. So at least one coder
# does the BOTTOM-UP pass FIRST, from construct definitions alone, and only
# afterwards are their independent boundary rulings compared with the pinned ones.
# Where the two diverge, the divergence is a FINDING about where the construct
# boundary actually sits, not coder error to be trained away.
#
#   BOTTOM-UP      construct definitions only. NO pinned token decisions, NO
#                  example anchors, NO numeric targets. Written out below.
#   FULL CODEBOOK  definitions plus anchors plus pinned routing. It is the
#                  language-model rubric, and it goes to the OTHER coder.
#
# Note what the bottom-up variant still spells out, because these three cannot be
# left to a reader's instinct without changing what is measured: the multi-label
# instruction (several facets may be 1; do not pick a winner), the prohibition on
# inferring a facet from the crime rather than from the text, and the stance rule
# -- negation and stance are delegated to the humans and to the language models
# precisely where the word list disclaims them (design section 6).
#
# The four-way principled distinction is stated in the terms that keep it
# testable: desert is the FACT that punishment is owed, proportionality is HOW
# MUCH fits, censure is the EXPRESSIVE function, moral balance is the DEBT/ledger
# metaphor. The coding has to be able to fire desert=1 with proportionality=0 on
# one sentence and the reverse on another, which is what makes the text-side H6
# dissociation (E5) measurable at all.

In [ ]:
# === DO (2 of 17): the bottom-up codebook, written out ===
CONSTRUCT_DEFINITIONS = """
BOTTOM-UP CODEBOOK (definitions only; NO pinned tokens, NO example anchors).
Score each facet 0 (absent) or 1 (clearly present) [or 0-2 intensity]; several may
be 1 at once. Do NOT pick one winner. Do NOT infer a facet from the crime. Handle
negation/stance: "I do NOT want him to suffer" -> suffering 0; a passage calling
deserving-punishment "just revenge" to REJECT it critiques desert, does not endorse it.

  DESERT           the OFFENDER earned/merited punishment as a backward fact about
                   what they did or are (agent-focused, magnitude-unbounded).
  PROPORTIONALITY  the PUNISHMENT's calibration/amount -- a bounded fit, an explicit
                   ceiling/floor ("fit the crime", "not too harsh/lenient").
  CENSURE          the social MESSAGE -- condemnation/accountability, satisfiable
                   symbolically ("send a message it's wrong", "society condemns").
  MORAL_BALANCE    restoring a LEDGER -- righting the wrong, paying a debt, closure
                   (distinct from sizing: about closure, not calibration).
  REVENGE          retaliation, getting even, symmetric hitting-back.
  SUFFERING        wanting pain beyond any instrumental aim.
  DETERRENCE       preventing future crime by warning/example.
  INCAPACITATION   protecting society by removing/containing the offender.
  REHABILITATION   reform, treatment, second chance.
  holistic:        one of principled | vengeful | both | neither.
"""

CODEBOOK_OUT = "codebook_bottom_up.txt"
with open(CODEBOOK_OUT, "w") as fh:
    fh.write(CONSTRUCT_DEFINITIONS)
print(f"Wrote {CODEBOOK_OUT} (definitions only; the disjoint-anchor variant).")

In [ ]:
# === DO (3 of 17): what to send out, and what has to come back ===
# On pass B this cell only prints; it changes nothing.
if not HAVE_CODERS:
    print("=" * 72)
    print("PASS A -- nothing to adjudicate yet. Here is the whole procedure.")
    print()
    print("  0. POST THE OSF AMENDMENT FIRST. Prereg 6.7 prescribes no human")
    print("     coding. The codebook, the coder instructions, the reliability")
    print("     statistics, the human-match thresholds and the frozen split all")
    print("     go up as one dated amendment BEFORE a coder reads a single")
    print("     response. This is the step that cannot be done afterwards.")
    print("  1. Send strata_sample.csv and codebook_bottom_up.txt to two or more")
    print("     trained coders. At least ONE codes bottom-up FIRST -- keep the")
    print("     language-model anchor set out of that coder's materials, so any")
    print("     agreement cannot be an artifact of shared exemplars.")
    print("  2. Each coder returns ONE filled CSV: coder_A.csv, coder_B.csv, ...")
    print("     They fill the nine facet columns, the holistic column, coder_id,")
    print("     and notes. Everything else comes back untouched.")
    print("  3. HAZARD: response_id and item_type must come back EXACTLY as they")
    print("     went out. Every alignment below keys on that pair; a re-sorted or")
    print("     re-typed id column shrinks the aligned set silently instead of")
    print("     raising, and the reliability is then computed on whatever rows")
    print("     happened to survive.")
    print("  4. Put the returned files in the working folder and re-run this")
    print("     notebook from the top.")
    print()
    print("  Nothing downstream is waiting on any of this. The registered gate")
    print("  clauses B and C run without it.")
    print("=" * 72)
else:
    print(f"PASS B -- {len(coder_files)} coder file(s) in the folder. Reliability,")
    print("adjudication and (where the ensemble has been scored) clause A follow.")

In [ ]:
# === DO (4 of 17): reading a coder's numbers, and the two helpers ===
def _to_num(s):
    # errors='coerce' turns a blank cell, a stray note, or a coder's "?" into NaN
    # rather than raising. That matters: an unfilled cell must read as MISSING,
    # never as a 0, or an unread row would be scored as unanimous agreement that
    # the facet is absent.
    return pd.to_numeric(s, errors="coerce")

def _ac1_se(a, b, cats, q, pk, pe, ac1):
    """The analytic standard error that goes beside AC1 (Gwet 2008 approximation).

    A standard error says how much a coefficient would move on a different sample
    of rows. The per-unit chance term is held at its overall value rather than
    recomputed per row, so this is a first-order approximation to the linearized
    Gwet variance. It is REPORT-ONLY: no gate reads it. The gate that does use an
    interval -- clause A's lower-bound rule -- is computed further down on a
    bootstrap, which makes no such approximation.
    """
    try:
        n = len(a)
        pa_i = (a == b).astype(float)
        pe_i = np.array([sum(pk[c] * (1 - pk[c]) for c in cats) / (q - 1)] * n)
        num = pa_i - 2 * (1 - ac1) * pe_i
        # The only way out of this expression that is not a number is NaN, from the
        # (1 - pe) == 0 branch: np.var is non-negative by construction, n is at
        # least 2 because the caller drops out below that, and (1 - pe)**2 is
        # positive wherever it is not zero. So var == var, the not-NaN test, is the
        # whole guard. A test for var >= 0 beside it would describe a branch that
        # cannot be reached, which is worse than no test at all: it reads like a
        # protection and protects nothing.
        var = np.var(num, ddof=1) / n / (1 - pe) ** 2 if (1 - pe) != 0 else np.nan
        return np.sqrt(var) if var == var else np.nan
    except Exception:
        return np.nan

def cohen_kappa_2(y1, y2):
    """Cohen's kappa between two coders: the classic chance-corrected agreement.

    Reported for reference alongside AC1 so a reader can see the paradox
    directly: on a sparse facet kappa sits far below AC1 on the very same rows.
    Computed inline because pingouin ships no two-rater helper.
    """
    m = pd.DataFrame({"a": _to_num(y1), "b": _to_num(y2)}).dropna()
    if len(m) < 2:
        return np.nan
    a = m["a"].astype(int).values
    b = m["b"].astype(int).values
    cats = sorted(set(a).union(set(b)))
    # Chance agreement from the PRODUCT of the two coders' marginals -- the term
    # that inflates toward the observed agreement when one category dominates.
    po = np.mean(a == b)
    pe = sum((np.mean(a == c)) * (np.mean(b == c)) for c in cats)
    return (po - pe) / (1 - pe) if (1 - pe) != 0 else np.nan

In [ ]:
# === DO (5 of 17): Gwet's AC1, the human-side statistic of record ===
def gwet_ac1_irr(y1, y2):
    """Gwet's AC1 between two coders, with its analytic standard error.

    An agreement coefficient: how far two raters agree beyond what agreeing at
    random would produce. Returns (ac1, se). Works for 0/1 and for 0-2 coding.

    WHY AC1 RATHER THAN KAPPA IS THE PRIMARY STATISTIC HERE (design section 8).
    Cohen's kappa divides the excess over chance by (1 - chance), and builds its
    chance term from the product of the two raters' marginals. When one category
    dominates -- revenge fires on well under 5% of responses -- that product
    approaches the observed agreement, the denominator collapses, and two coders
    who agree on 99% of rows can post a kappa near zero. That is the kappa
    paradox, and it would read as "humans cannot code revenge" when what happened
    is "revenge is rare". Gwet's chance term is pk(1-pk) summed over categories:
    largest when the categories are balanced, small at the extremes. It does not
    punish prevalence imbalance, which is exactly the property the sparse facets
    need. Kappa is reported beside it for reference, not as the decision statistic.
    """
    m = pd.DataFrame({"a": _to_num(y1), "b": _to_num(y2)}).dropna()
    # Fewer than two usable rows: no reliability to estimate.
    if len(m) < 2:
        return (np.nan, np.nan)
    a = m["a"].astype(int).values
    b = m["b"].astype(int).values
    # The observed category set, so a 0/1 facet and a 0-2 intensity facet both work.
    cats = sorted(set(a).union(set(b)))
    q = len(cats)
    if q < 2:
        # ONE CATEGORY IN THE WHOLE COLUMN -- typically two coders who both marked
        # the facet absent on every row. There is no agreement to estimate: Gwet's
        # chance term divides by (q - 1), which is zero here, and any number put in
        # its place is a convention rather than a measurement. Returning 1.00 with a
        # standard error of 0.00 would make the facet nobody could see read as the
        # best-measured facet in the table, which is what design section 7 warns
        # about for proportionality at its ~1% prevalence. NaN is the honest answer,
        # and DO (7) turns it into the words "not estimable".
        return (np.nan, np.nan)
    pa = np.mean(a == b)           # observed agreement
    # Chance agreement per Gwet: pe = (1/(q-1)) * sum pk(1-pk), with pk the
    # category's marginal averaged over the two coders.
    pk = {c: (np.mean(a == c) + np.mean(b == c)) / 2.0 for c in cats}
    pe = sum(pk[c] * (1 - pk[c]) for c in cats) / (q - 1)
    ac1 = (pa - pe) / (1 - pe) if (1 - pe) != 0 else np.nan
    return (ac1, _ac1_se(a, b, cats, q, pk, pe, ac1))

In [ ]:
# === DO (6 of 17): load the coder files, one rater per file ===
# Each returned file is ONE rater. Two things quietly turn two raters into one, or
# one rater into a doubled vote, and both are checked here rather than assumed:
#
#   a repeated coder_id   two coders who both typed 'A' would collide onto one
#                         dictionary key and one of them would be discarded with no
#                         message at all -- and DO (7) would then fail on an index
#                         error looking for a second rater. Ids are made unique by
#                         appending the file name, so a collision costs one printed
#                         line and never a coder.
#   a repeated row        a double-entered response_id x item_type widens that
#                         row's vote in the adjudication below, multiplies it in
#                         every alignment, and travels into gold_codes.csv where
#                         notebook 04 would embed the same sentence twice. There is
#                         no way to tell which of the two entries the coder meant,
#                         so a file with one is set aside whole and named.
#
# Nothing here raises. A coder file is not a required input -- pass A is a valid
# run -- so a defective one costs that coder, is reported in one line, and the
# notebook carries on with whoever is left.
coders, cids = {}, []
if HAVE_CODERS:
    _set_aside = []
    for cf in coder_files:
        cdf = pd.read_csv(cf)
        if not set(KEY).issubset(cdf.columns):
            _set_aside.append(f"{cf}: no response_id / item_type column, so it "
                              "cannot be aligned with anything")
            continue
        _d = cdf[cdf.duplicated(KEY, keep=False)]
        if len(_d):
            _set_aside.append(
                f"{cf}: {len(_d)} rows share a response_id x item_type, e.g. "
                f"{_d[KEY].iloc[0].to_dict()}. Ask the coder for one row per "
                "response, then re-run.")
            continue
        # The coder's own id from inside the file, falling back to the file name
        # where the column was left blank.
        cid = (str(cdf["coder_id"].dropna().iloc[0])
               if "coder_id" in cdf.columns and cdf["coder_id"].notna().any()
               else cf)
        if cid in coders:
            print(f"  coder_id '{cid}' appears in more than one file. Both are "
                  f"kept; this one is recorded as '{cid} ({cf})'.")
            cid = f"{cid} ({cf})"
        coders[cid] = cdf.set_index(KEY)
    for _msg in _set_aside:
        print(f"  SET ASIDE -- {_msg}")
    cids = list(coders.keys())
    print(f"Loaded {len(cids)} usable coder file(s): {cids}")
    if len(cids) < 2:
        # Reported, not raised. Reliability is a two-rater question, so with fewer
        # than two usable files there is nothing to adjudicate and the notebook
        # finishes the way it does on pass A.
        HAVE_CODERS = False
        NOT_RUN_REASON = NOT_RUN_REASON or (
            f"only {len(cids)} usable coder file(s) after the checks in DO (6)")
        print("  Fewer than two usable coder files, so no reliability, no")
        print("  adjudication and no clause A. This is pass A with extra reading.")

In [ ]:
# === DO (6 of 17, cont.): align the coders on the rows all of them read ===
# Intersection, so every statistic below rests on rows that every coder actually
# read. A sharp drop from the sample's row count means a coder dropped rows or
# re-typed an id column, and the printed number is where that becomes visible --
# the alignment would otherwise shrink silently and the reliability would be
# computed on whatever rows happened to survive.
#
# Each coder frame is already unique on response_id x item_type (DO (6) sets aside
# any file that is not), so this intersection is unique too, and gold_codes.csv is
# built one row per index entry. That is the chain by which the one-row-per-key
# contract reaches the file this notebook writes; DO (17) and SAVE check it again
# on the frames themselves rather than trusting the chain.
if HAVE_CODERS:
    common_idx = None
    for cid in cids:
        idx = coders[cid].index
        common_idx = idx if common_idx is None else common_idx.intersection(idx)
    print(f"Aligned on {len(common_idx)} common response_id x item_type rows, "
          f"out of {len(template)} sent out.")
    if len(common_idx) < 0.8 * len(template):
        print("  That is well below what went out. Check that response_id and")
        print("  item_type came back exactly as they were sent, in every file.")
else:
    print("No usable coder files: no reliability, no adjudication, no clause A.")

In [ ]:
# === DO (7 of 17): per-facet reliability between the coders ===
# Computed on the FIRST TWO coders. Adjudication across all of them happens
# further down; AC1 and kappa are two-rater coefficients, and reporting them on a
# named pair is honest in a way that averaging over pairs is not.
irr_rows = []
if HAVE_CODERS:
    # HAVE_CODERS is True only where DO (6) found two or more USABLE coder
    # files -- it turns itself off below two, so cids[0] and cids[1] both
    # exist here. Counting files rather than raters is what would break this.
    c0, c1 = cids[0], cids[1]
    for f in FACET_NAMES:
        # A facet a coder never filled in is skipped, not scored as a column of
        # zeros: silence is missing data, not agreement that the facet is absent.
        if f not in coders[c0].columns or f not in coders[c1].columns:
            continue
        y1 = coders[c0].loc[common_idx, f]
        y2 = coders[c1].loc[common_idx, f]
        ac1, ac1_se = gwet_ac1_irr(y1, y2)
        kappa = cohen_kappa_2(y1, y2)
        # n_pos counts rows where EITHER coder saw the facet. This is the number
        # that decides whether the coefficient means anything: with revenge and
        # suffering at 0.7-2.5% base rate, a coefficient computed on roughly 14
        # positives carries a 95% half-width of about +/-0.26, so a facet whose
        # true agreement is .40 can post a sample .60 on luck alone (design 7).
        # Here it is reported across the whole drawn sample; the n_pos >= 30 rule
        # is applied to the TEST half in clause A below.
        n_pos = int(((_to_num(y1) > 0) | (_to_num(y2) > 0)).sum())
        # THE VERDICT IS A WORD, NEVER A BARE BOOLEAN, because three different
        # things can be true of a facet and only two of them are a measurement:
        #   reliable        AC1 at or above the bar, on enough positive cases.
        #   hard            AC1 below the bar. Descriptive-only however the models
        #                   score it -- if humans cannot agree on a facet, no
        #                   automated match can rescue it (design section 8).
        #   not estimable   no coefficient at all, or too few positive cases for
        #                   one to be about anything. Two coders who both marked a
        #                   facet absent on every row have agreed about an empty
        #                   cell and demonstrated nothing about whether they can
        #                   recognise the facet when it appears. A boolean that
        #                   only asked "is AC1 below .60" would call that facet
        #                   reliable, and it is the facet with no evidence at all.
        # `ac1 == ac1` is the not-NaN test.
        if not (ac1 == ac1) or n_pos < HUMAN_IRR_MIN_POS:
            verdict = "not estimable"
        elif ac1 < HUMAN_IRR_MIN:
            verdict = "hard"
        else:
            verdict = "reliable"
        irr_rows.append({
            "facet": f,
            "gwet_ac1": round(ac1, 3) if ac1 == ac1 else np.nan,
            "ac1_se": round(ac1_se, 3) if ac1_se == ac1_se else np.nan,
            "cohen_kappa": round(kappa, 3) if kappa == kappa else np.nan,
            "n_pos": n_pos,
            "irr_verdict": verdict,
            "min_n_pos": HUMAN_IRR_MIN_POS,
            "coder_pair": f"{c0} x {c1}",
        })
    irr_df = pd.DataFrame(irr_rows)

In [ ]:
# === DO (8 of 17): how to read that table ===
# HOW TO READ IT: gwet_ac1 is the decision statistic, cohen_kappa the reference,
# n_pos the power behind both. A high AC1 sitting on a single-digit n_pos is not
# reliability -- it is two coders agreeing that a rare thing is absent -- and
# irr_verdict is where that reading is applied rather than left to the reader.
#
# WHERE THIS VERDICT TRAVELS: nowhere automatically. gold_irr_by_facet.csv is
# written for the OSF amendment and NO notebook in this chain reads it. Notebook
# 10 reads clause A's file and nothing else from here. So "descriptive-only
# because the humans could not agree" is a reading the write-up has to carry by
# hand, and it belongs in the limitations paragraph or it is not said at all.
if HAVE_CODERS and len(irr_df):
    print("Per-facet inter-rater reliability (Gwet AC1 primary; kappa reference):")
    print(irr_df.to_string(index=False, na_rep="--"))
    _hard = irr_df[irr_df["irr_verdict"] == "hard"]["facet"].tolist()
    _none = irr_df[irr_df["irr_verdict"] == "not estimable"]["facet"].tolist()
    if _hard:
        # censure and moral_balance are the expected members. Both are principled
        # facets with NO admissible survey validator either (prereg 5.15 names
        # only four facet-to-scale mappings), so they are descriptive-only on that
        # ground as well.
        print(f"\n  HARD FOR HUMANS (AC1 < {HUMAN_IRR_MIN:.2f}): {_hard}")
        print("  -> descriptive-only regardless of what the models do. Expect")
        print("     censure and moral_balance here.")
    if _none:
        print(f"\n  NOT ESTIMABLE (n_pos < {HUMAN_IRR_MIN_POS}, or no coefficient "
              f"defined): {_none}")
        print("  -> no human reliability evidence exists for these facets. That is")
        print("     not a pass and it is not a failure: nobody has measured them.")
        print("     Expect proportionality here, at its ~1% corpus prevalence.")
elif HAVE_CODERS:
    print("No facet column was filled in by both of the first two coders.")
else:
    print("Reliability: not run.")

In [ ]:
# === DO (9 of 17): the holistic call, and the divergence log ===
# Krippendorff's alpha is an agreement coefficient for LABELS. NOMINAL, not
# ordinal: the four calls are unordered categories. "both" is not a midpoint
# between principled and vengeful -- it is a different claim about the response --
# so treating the codes as ordered would invent a distance that is not there. The
# integers below are labels only.
HOLISTIC_MAP = {lvl: i for i, lvl in enumerate(HOLISTIC_LEVELS)}
alpha_hol = None
if HAVE_CODERS:
    hol = []
    for cid in cids:
        col = (coders[cid].loc[common_idx, "holistic"]
               if "holistic" in coders[cid].columns else None)
        if col is not None:
            # An unrecognised or blank call maps to NaN, which krippendorff treats
            # as missing rather than as a fifth category.
            hol.append([HOLISTIC_MAP.get(str(v).strip().lower(), np.nan)
                        for v in col])
    if len(hol) >= 2:
        # Unlike the per-facet AC1 above, this uses ALL coders, not just the first
        # two. SCOPE: the holistic call is human-side reliability only. It is
        # barred from the E4 evidence family (anti-circularity rule 6) because it
        # is produced in the same pass by the same reader as the facet vector, so
        # it cannot corroborate facet-level co-occurrence.
        alpha_hol = krippendorff.alpha(
            reliability_data=np.array(hol, dtype=float),
            level_of_measurement="nominal")
        print(f"Holistic Krippendorff alpha (nominal): {alpha_hol:.3f}")
    else:
        print("Holistic alpha: not run (fewer than two coders had the column).")

    # The payoff of the two-variant coder materials. A coder who codes bottom-up
    # and then reads the pinned codebook will disagree with it somewhere; those
    # places are where the construct boundary is genuinely contested, and
    # recording them as findings is the difference between measuring the boundary
    # and asserting it.
    print("\nRecord bottom-up vs. codebook BOUNDARY divergences as FINDINGS")
    print("  (the 'notes' column, where a coder flags a place their bottom-up call")
    print("  differs from the pinned codebook). Construct findings, NOT coder error.")
else:
    print("Holistic alpha: not run.")

In [ ]:
# === DO (10 of 17): the tie rule, stated rather than inherited ===
# Adjudication is a VOTE COUNT, and with two coders every disagreement is an exact
# tie -- there is no majority to find. A tie is not a consensus, so the shipped
# rule writes the consensus cell as MISSING and marks <facet>_tie, which keeps a
# split call out of the criterion instead of resolving it by fiat in either
# direction. The two directional alternatives are implemented beside it so the
# team can change the policy in one place, and so the choice is visible in the
# code rather than buried in a rounding rule:
#
#   'unresolved'  tie -> missing, and the row drops out of that facet's downstream
#                        use. This is what ships.
#   'present'     tie -> the higher call, maximizing recall of a sparse facet
#   'absent'      tie -> the lower call, the conservative reading
#
# Whichever is in force, <facet>_tie carries the count, so a reader can see how
# much of the criterion rests on a split call, and a split call can never be read
# back later as an agreed one.
TIE_RULE = "unresolved"      # 'unresolved' | 'present' | 'absent'
print(f"TIE_RULE = '{TIE_RULE}'.")

In [ ]:
# === DO (11 of 17): adjudicate the coders into one consensus per row ===
if HAVE_CODERS:
    rows = []
    for idx in common_idx:
        rid, it = idx
        rec = {"response_id": rid, "item_type": it}
        for f in FACET_NAMES:
            # Skip a coder who has no column for this facet rather than reading a
            # NaN into the vote.
            vals = [_to_num(coders[c].loc[idx, f]) for c in cids
                    if f in coders[c].columns]
            # v == v drops NaN (an unfilled cell), so a blank does not drag the vote.
            vals = [v for v in vals if v == v]
            if vals:
                # A counted vote, not a rounded mean. Counting is what makes a
                # split call visible: the top two vote counts are compared, and
                # where they are equal there is no majority and TIE_RULE decides
                # what the consensus cell carries. Works unchanged for 0/1 and for
                # 0-2 intensity coding.
                votes  = Counter(int(round(v)) for v in vals)
                ranked = votes.most_common()
                tied   = len(ranked) > 1 and ranked[0][1] == ranked[1][1]
                if tied:
                    rec[f] = (np.nan     if TIE_RULE == "unresolved" else
                              max(votes) if TIE_RULE == "present"    else
                              min(votes))
                else:
                    rec[f] = ranked[0][0]
                # The tie marker travels with the row whatever TIE_RULE is set to.
                rec[f + "_tie"] = int(tied)
                # 1 wherever the coders' calls were not all identical: the
                # row-level record of where the construct boundary was contested.
                # With exactly two coders this coincides with _tie; with three or
                # more it does not, because 2-1 is a disagreement that still has a
                # majority.
                rec[f + "_disagree"] = int(len(votes) > 1)
            else:
                rec[f] = np.nan
                rec[f + "_disagree"] = 0
                rec[f + "_tie"] = 0
        rows.append(rec)
    print(f"Adjudicated {len(rows)} rows across {len(cids)} coders.")

In [ ]:
# === DO (12 of 17): the holistic majority, and the text that travels with it ===
if HAVE_CODERS:
    for rec in rows:
        idx = (rec["response_id"], rec["item_type"])
        # Counter.most_common breaks a tie by insertion order, i.e. by coder order
        # in `cids`. With an even number of coders and a genuine split the first
        # coder's call wins -- the holistic column is descriptive coverage only and
        # feeds no gate (anti-circularity rule 6), so the tie rule carries no
        # weight here and is not worth a policy.
        hvals = [str(coders[c].loc[idx, "holistic"]).strip().lower()
                 for c in cids if "holistic" in coders[c].columns]
        hvals = [h for h in hvals if h and h != "nan"]
        rec["holistic"] = Counter(hvals).most_common(1)[0][0] if hvals else ""
        # The text travels with the consensus codes because the reference-pool
        # notebook builds embedding passages straight out of it; the stratum
        # travels because human-match is reported BY STRATUM (design section 4) --
        # agreement on the random slice and agreement on the survey-disagreement
        # slice are different claims and must not be pooled. .iloc[0] is safe here
        # only because LOAD (1 of 3) asserted the template is one row per key.
        trow = template[(template.response_id == rec["response_id"])
                        & (template.item_type == rec["item_type"])]
        rec["text"]    = trow["text"].iloc[0] if len(trow) else ""
        rec["stratum"] = trow["stratum"].iloc[0] if len(trow) else ""
    gold_codes = pd.DataFrame(rows)

    # Attach the frozen SEED/TEST label. The split was decided once, in notebook
    # 03, from RANDOM_SEED and before any scoring; it is READ here, never
    # recomputed, so no re-run of this notebook can move a row from one half to
    # the other after the results are visible.
    #
    # THE TWO CHECKS AROUND THE MERGE. A left merge multiplies rows wherever the
    # RIGHT frame holds more than one row per key, and it does so silently: three
    # duplicated split rows would make three copies of a gold row, every count
    # printed below would still look plausible, and clause A would then be computed
    # on a sample that is not the sample. The first check is on the right frame
    # before the merge; the second is on the result, and it is the one that still
    # stands if this cell is ever run against a `split` that LOAD did not build.
    assert not split.duplicated(KEY).any(), (
        "gold_split_assignment.csv holds more than one row for "
        + str(split[split.duplicated(KEY, keep=False)][KEY].head(4)
              .to_dict("records"))
        + ". Merging it would multiply the coded rows. Re-run notebook 03.")
    _before = len(gold_codes)
    gold_codes = gold_codes.merge(
        split[["response_id", "item_type", "gold_split"]],
        on=KEY, how="left")
    assert len(gold_codes) == _before, (
        f"attaching the SEED/TEST split turned {_before} adjudicated rows into "
        f"{len(gold_codes)}. The split file is not one row per response_id x "
        "item_type, and every number below would be computed on inflated data.")

    _unassigned = int(gold_codes["gold_split"].isna().sum())
    if _unassigned:
        print(f"  NOTICE: {_unassigned} coded rows are not in the split file. They")
        print("  carry no SEED/TEST label and are excluded from clause A below.")
    print(gold_codes["gold_split"].value_counts().to_string())

In [ ]:
# === DO (13 of 17): how much of the criterion rests on a split call ===
# Printed beside the consensus because it is the number that says how far the
# criterion can be trusted, per facet. A facet whose ties run into the dozens is
# one where the coders did not converge -- the AC1 table will say so too -- and
# clause A for that facet is then computed on rows the humans could not settle.
if HAVE_CODERS and gold_codes is not None:
    _tie_cols = [f + "_tie" for f in FACET_NAMES if f + "_tie" in gold_codes.columns]
    if _tie_cols:
        _ties = gold_codes[_tie_cols].sum().astype(int)
        _ties.index = [c[:-4] for c in _ties.index]
        print(f"Split calls per facet (TIE_RULE = '{TIE_RULE}'), out of "
              f"{len(gold_codes)} rows:")
        print(_ties.to_string())
        print("  Recorded as ties, never as agreed absences. ACTION: adjudicate")
        print("  them -- a third reader, or the two coders in discussion -- and")
        print("  refill the facet cell, or report that facet on its resolved rows")
        print("  only.")
else:
    print("Consensus: not run.")

In [ ]:
# === DO (14 of 17): clause A, part 1 -- the agreement coefficient ===
# WHAT CLAUSE A IS, AND WHOSE IT IS. Prereg 6.7 states a TWO-clause gate: the
# models agree at ICC >= .60, and the facet correlates r >= .30 with its matching
# survey scale. Those two are computed unchanged elsewhere in the chain. Clause A
# -- ensemble against human coders -- and the gold standard it runs on are an
# ADDITION, carried on the OSF amendment. It adds to that gate and never relaxes it.
#
# WHY IT HAS TO EXIST. Prereg 6.7's own words: "Agreement among the models is
# reliability, not validity... Models trained on similar data can share the same
# bias, and the line between principled desert and revenge is the thing this study
# examines, so we do not treat the ensemble as ground truth." Three models
# agreeing on the wrong reading produce a high ICC and a tight ensemble mean. The
# only thing that can tell that apart from accuracy is agreement with humans on
# responses the models had no hand in selecting.
def gwet_ac1(a, b):
    """Gwet's AC1 for two BINARY raters -- the form clause A calls.

    Same shape as kappa -- (observed - chance) / (1 - chance) -- with a chance
    term that does not blow up under prevalence imbalance: pe = 2*pi*(1-pi), with
    pi the mean of the two raters' prevalences. This is why it is the estimator of
    record for revenge and suffering (design section 7).

    Kept separate from gwet_ac1_irr above rather than merged with it: that one
    returns a standard error and reads pandas columns that may hold blanks, this
    one takes two clean binary arrays and is called several thousand times inside
    the bootstrap. One function doing both jobs would be slower and less clear.
    """
    a = np.asarray(a); b = np.asarray(b)
    n = len(a)
    if n == 0:
        return np.nan
    pa = np.mean(a == b)
    pi = (np.mean(a) + np.mean(b)) / 2.0
    pe = 2 * pi * (1 - pi)
    # WHAT THE (1 - pe) GUARD ACTUALLY COVERS, since the arithmetic is easy to
    # misread: pe reaches its maximum of 1 when the two raters' mean prevalence is
    # exactly .5, and the guard is what keeps that from dividing by zero. It is NOT
    # a guard on the raters agreeing about everything. Two raters who both call
    # every row positive give pi = 1 and pe = 0, and this function returns 1.00 --
    # arithmetic rather than reliability. That case is caught in _agree_point
    # below, before this function is ever reached.
    return (pa - pe) / (1 - pe) if (1 - pe) != 0 else np.nan

In [ ]:
# === DO (14 of 17, cont.): one point estimate, chosen by the data ===
def _agree_point(ens, hum):
    """Ensemble-vs-human agreement, one point estimate.

    Two estimators, because the nine facets are not one kind of variable:
      ICC(2,1)     when both sides are genuinely graded (more than two distinct
                   values each). Two-way random, absolute agreement -- it
                   penalises a constant offset, which is exactly what a model
                   that reads every passage as slightly more retributive than the
                   coders did would produce, and which a plain correlation would
                   forgive.
      Gwet's AC1   otherwise, on the binarized version: the ensemble counts the
                   facet present above ENS_PRESENT, the coder counts it present
                   at any nonzero code. The sparse facets sit at 0.7-2.5%
                   prevalence, where kappa collapses toward zero even at
                   near-perfect agreement because chance agreement is estimated
                   as almost total (design sections 7 and 8).

    Returns NaN on fewer than 5 usable pairs, on a constant human column, or where
    both raters come out constant after thresholding -- no agreement coefficient is
    defined against a rater who never varies.
    """
    d = pd.DataFrame({"ens": ens, "hum": hum}).dropna()
    # Pairwise-complete: a row is used only where both raters produced a value.
    if len(d) < 5 or d["hum"].nunique() < 2:
        return np.nan
    # Route to ICC only when both sides are actually graded. The ensemble mean is
    # continuous, but a facet the coders scored 0/1 has two levels on the human
    # side, and an ICC of a binary rater against a continuous one is not the
    # coefficient anyone means; fall through to AC1.
    #
    # pingouin wants long form, so the pair is stacked into target/rater/score with
    # a synthetic target id per row. ICC2 is the single-rater absolute-agreement
    # form -- the right one here, since the question is whether ONE ensemble
    # reading matches ONE human consensus reading.
    if d["ens"].nunique() > 2 and d["hum"].nunique() > 2:
        try:
            long = pd.DataFrame({
                "target": list(range(len(d))) * 2,
                "rater": ["ens"] * len(d) + ["hum"] * len(d),
                "score": list(d["ens"]) + list(d["hum"]),
            })
            icc = pg.intraclass_corr(data=long, targets="target", raters="rater",
                                     ratings="score")
            row = icc[icc["Type"] == "ICC2"]
            return float(row["ICC"].iloc[0]) if len(row) else np.nan
        except Exception:
            pass
    # THE PRESENCE THRESHOLD IS THE WHOLE COMPARISON. The ensemble mean is a
    # continuous 0-to-1 number and the human code is 0/1 or 0-2, so one of them has
    # to be brought onto the other's scale before an agreement coefficient means
    # anything. ENS_PRESENT is that rule and it is stated in the setup cell, not
    # buried here, because the estimate the gate reads moves with it.
    a = (d["ens"] > ENS_PRESENT).astype(int).values
    b = (d["hum"] > 0).astype(int).values
    # Both raters constant after thresholding: they agree about everything and have
    # shown nothing. gwet_ac1 returns 1.00 there on the arithmetic alone, and a 1.00
    # built on an empty cell would clear clause A outright. Undefined is honest.
    if len(set(a)) < 2 and len(set(b)) < 2:
        return np.nan
    return gwet_ac1(a, b)

In [ ]:
# === DO (14 of 17, cont.): the bootstrap interval the gate reads ===
def bootstrap_agreement(ens, hum, n_boot=2000, seed=7):
    """Bootstrap 95% confidence interval around the agreement point estimate.

    Resamples the TEST rows with replacement and recomputes the agreement each
    time, so the interval reflects how much the estimate moves when the particular
    responses that landed in TEST change. Percentile method, 2000 resamples --
    enough for a stable 2.5/97.5 percentile without making the cell slow, since
    each resample may re-run an ICC. seed=7 fixes the resampling so the reported
    bound is reproducible from the posted data.

    Returns (estimate, lo, hi, n_ok). The estimate is the preregistered quantity
    and is reported; `lo` is what the gate decides on; `n_ok` is how many of the
    n_boot draws came back with a number.

    WHY n_ok IS RETURNED AND REPORTED. A draw that comes back degenerate -- an
    all-constant human column, too few pairs -- returns NaN and is skipped rather
    than counted as zero, which would drag the lower bound down artificially. But
    the skipped draws are NOT a random subset: they are precisely the draws that
    caught fewer positive cases, so the surviving draws are the ones with more
    variation in the human column and the interval is narrower than the interval
    on all n_boot would be. At about 1% prevalence roughly one draw in seven can
    come back degenerate. The count says how much of the interval that is, and a
    reader cannot see it if it is not printed.

    HAZARD: this is the RESAMPLING seed only. It has nothing to do with the frozen
    SEED half of the sample, and re-rolling it re-rolls a reported interval.
    """
    rng = np.random.default_rng(seed)
    ens = np.asarray(ens, dtype=float); hum = np.asarray(hum, dtype=float)
    mask = ~(np.isnan(ens) | np.isnan(hum))
    ens, hum = ens[mask], hum[mask]
    est = _agree_point(ens, hum)
    if len(ens) < 5:
        return est, np.nan, np.nan, 0
    # Resample rows, not facets: a bootstrap draw is a hypothetical alternative
    # TEST half of the same size.
    #
    # THE ONLY PLACE THIS NOTEBOOK SILENCES A WARNING, and it is scoped to this
    # loop with catch_warnings so it is undone the moment the loop ends. A draw
    # that catches no positive cases makes numpy divide by zero, and 2,000 draws
    # on a 1%-prevalence facet can produce thousands of identical notices about a
    # condition this loop already handles by skipping the draw. Everywhere else in
    # the session warnings are left alone, because a warning nobody sees is a
    # finding nobody has.
    boots = []
    idx = np.arange(len(ens))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for _ in range(n_boot):
            s = rng.choice(idx, size=len(idx), replace=True)
            v = _agree_point(ens[s], hum[s])
            if not (v is None or np.isnan(v)):
                boots.append(v)
    if not boots:
        return est, np.nan, np.nan, 0
    # Percentile interval. No normal approximation: at these base rates the
    # bootstrap distribution of an agreement coefficient is skewed and bounded, and
    # a symmetric interval would put the lower bound in the wrong place -- which is
    # the one number the gate reads.
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return est, lo, hi, len(boots)

In [ ]:
# === DO (15 of 17): the TEST half, joined to the ensemble ===
# WHY TEST ONLY, ALWAYS. The drawn sample has three jobs -- validity criterion,
# embedding reference passages, word-list expansion -- and notebook 03 froze it
# into halves before any scoring so those jobs cannot contaminate each other. SEED
# supplies the rubric anchors, the reference passages and the lexicon expansions;
# TEST supplies every gate and convergence number. No gate figure may be computed
# on responses that seeded the stream it is checking, or part of the agreement
# would be true by construction (design section 8).
CLAUSE_A_RUNS = bool(HAVE_CODERS and ENSEMBLE_AVAILABLE and gold_codes is not None)
gold_test = None
if CLAUSE_A_RUNS:
    # Inner join, so a gold row the ensemble never scored -- a text below the word
    # floor, a response that failed on every leg -- drops out of the comparison
    # rather than entering it as a missing value. Clause A is about agreement where
    # both raters spoke.
    #
    # WHAT THE PRINTOUT MEANS: the number of TEST rows the gate actually runs on.
    # If it is far below half the drawn sample, the join is failing (a key
    # mismatch, or the split did not attach) and every number below would rest on
    # a handful of rows.
    #
    # THE TWO CHECKS AROUND THE MERGE. ens_wide is built by a pivot indexed on KEY
    # in LOAD (3 of 3), which is what makes it one row per response_id x item_type
    # -- pandas raises rather than fan out. That guarantee is re-stated here as an
    # assertion because the two cells can be run apart and a later edit to the
    # reshape would move the guarantee without moving this merge. An inner join can
    # only drop rows, never add them, so the second check is that the count did not
    # RISE: a rise is a fan-out, and it would inflate every clause A sample size
    # while the printed line still looked right.
    assert not ens_wide.duplicated(KEY).any(), (
        "the wide ensemble frame holds more than one row for some response_id x "
        "item_type, so joining it would multiply the gold TEST rows. Re-run "
        "LOAD (3 of 3).")
    _left = gold_codes[gold_codes["gold_split"] == "TEST"]
    gold_test = _left.merge(ens_wide, on=KEY, how="inner")
    assert len(gold_test) <= len(_left), (
        f"joining the ensemble turned {len(_left)} gold TEST rows into "
        f"{len(gold_test)}. An inner join cannot add rows unless a key repeats on "
        "one side, so the ensemble frame is not one row per response_id x "
        "item_type and clause A would be computed on inflated data.")
    print(f"Gold TEST rows with ensemble scores: {len(gold_test)} "
          f"(of {len(_left)} gold TEST rows)")
else:
    print(f"Clause A: not run ({NOT_RUN_REASON}).")


In [ ]:
# === DO (15 of 17, cont.): where the human side of each facet lives ===
# THE ONE PLACE the human-side column naming is decided. The adjudication above
# writes the consensus under the plain facet name, so this is the identity
# function; point it at a suffixed name if a coder export uses one. Centralised
# deliberately -- a naming mismatch here would make every facet report a missing
# human side and the whole clause silently vacuous, which is why that case is
# recorded as an explicit FAIL row below rather than quietly skipped.
def HUMAN_COL(f):
    return f          # e.g. 'desert'; change to f'{f}_gold' if needed

def human_side_missing(f):
    """Is this facet's human side absent? Returns the reason, or "" if it is there.

    TWO WAYS IT CAN BE ABSENT, and checking only the first is what makes this
    check useless. (1) The column is not in the frame at all. (2) The column IS
    there and holds no number in any row. The second is what happens when the
    coder files use a different column name from HUMAN_COL: the adjudication in
    DO (11) writes a cell for every facet in FACET_NAMES whether or not any coder
    supplied it, so a renamed facet arrives here as a full column of blanks with
    exactly the right name. Testing only for the name would then let nine
    plumbing faults through the gate dressed as nine facets that failed against
    human coders -- opposite readings of the same table.
    """
    hc = HUMAN_COL(f)
    if hc not in gold_test.columns:
        return "no human column"
    if f"{f}_ens_mean" not in gold_test.columns:
        return "no ensemble column"
    if not pd.to_numeric(gold_test[hc], errors="coerce").notna().any():
        return "human column is empty"
    return ""

PLUMBING_NOTES = {"no human column", "no ensemble column",
                  "human column is empty"}

In [ ]:
# === DO (15 of 17, cont.): clause A per facet, on the TEST half ===
# A facet passes only if BOTH hold:
#
#   ci_lo >= .60   Gate on the LOWER BOUND, not the point estimate. This is the
#                  tightening registered in MEASUREMENT_DESIGN.md section 0.
#                  Prereg 6.7 states .60 as a threshold on the estimate; with
#                  revenge and suffering at 0.7-2.5% prevalence and a half-split
#                  sample, the TEST half holds on the order of 14 positive cases
#                  for the hardest facets, where a 95% half-width runs to about
#                  +/-.26. A facet whose TRUE human-match is .40 can then post a
#                  sample .60 on sampling luck alone. Requiring the interval to
#                  clear .60 means the facet has to be good enough that even the
#                  unlucky end of the estimate is acceptable. The preregistered
#                  point estimate is reported in the same table, so the stricter
#                  decision never hides the registered quantity.
#
#   n_pos >= 30    A minimum number of POSITIVE human-coded cases. Agreement on a
#                  facet nobody expressed is agreement on the all-zero cell: two
#                  raters who both say "absent" 300 times look like perfect
#                  agreement and have demonstrated nothing about whether they can
#                  recognise the facet when it appears (design section 7).
#
# A failing facet carries NO primary E3-E7 claim however high its inter-model ICC.
# All of this stands INSIDE the exploratory family: clause A decides gate-backed
# versus descriptive-only, never confirmatory.
#
# hm_pooled IS EMPTIED HERE, not appended to. Changing CLAUSE_A_MIN and re-running
# this block is the obvious thing to do, and against a list that only grew it would
# write a second set of pooled rows: notebook 10 finds two pooled rows for a facet
# and records clause A as "not run" for every facet, silently.
hm_pooled = []
if CLAUSE_A_RUNS:
    for f in FACET_NAMES:
        hc = HUMAN_COL(f)
        _absent = human_side_missing(f)
        if _absent:
            # NOT a result. A table of nine such rows is a plumbing fault -- the
            # join or HUMAN_COL is wrong -- and it is recorded loudly, in the
            # note, so it cannot pass for nine failed facets.
            hm_pooled.append({"scope": "pooled", "facet": f, "clause_a": "FAIL",
                              "n_pos": 0, "note": _absent})
            continue
        est, lo, hi, n_ok = bootstrap_agreement(gold_test[f"{f}_ens_mean"],
                                                gold_test[hc])
        n_pos = int((pd.to_numeric(gold_test[hc], errors="coerce") > 0).sum())
        # A NaN lower bound (too few pairs to bootstrap) is a FAIL: "could not be
        # estimated" is not "passed".
        passed = bool((not np.isnan(lo)) and lo >= CLAUSE_A_MIN
                      and n_pos >= CLAUSE_A_MIN_POS)
        hm_pooled.append({
            "scope": "pooled", "facet": f, "est": est, "ci_lo": lo, "ci_hi": hi,
            "ci_width": (hi - lo) if not np.isnan(hi) else np.nan,
            "n_pos": n_pos, "n_rows": int(len(gold_test)), "boot_ok": n_ok,
            "ens_present_at": ENS_PRESENT,
            "clause_a": "PASS" if passed else "FAIL",
            "note": "PASS" if passed else
                    f"FAIL (lo < {CLAUSE_A_MIN:.2f} or n_pos < {CLAUSE_A_MIN_POS})"})
    CLAUSE_A_STATUS = "run"

In [ ]:
# === DO (15 of 17, cont.): print the pooled clause A table ===
# WHAT THIS TABLE MEANS, column by column:
#   est       the preregistered point estimate (reported, not gated on)
#   ci_lo     the number the gate reads
#   ci_width  how much the estimate would move on a different TEST draw. A wide
#             interval that happens to clear .60 is a lucky pass, and printing the
#             width is what makes that visible rather than arguable.
#   n_pos     positive human cases behind the estimate
#   boot_ok   how many of the 2,000 resamples produced a number. Below 2,000 the
#             interval rests on the draws that happened to catch more positives.
#   clause_a  PASS = eligible to be gate-backed, subject to clauses B and C.
#             'not run' = never computed, which is not a failure.
#   note      'no human column', 'no ensemble column' and 'human column is empty'
#             all mean the join or HUMAN_COL is wrong, NOT that the facet failed
#             -- a table of nine such rows is a plumbing fault, not a result.
if CLAUSE_A_RUNS:
    _pooled = pd.DataFrame(hm_pooled)
    _show = [c for c in ("facet", "est", "ci_lo", "ci_hi", "ci_width", "n_pos",
                         "n_rows", "boot_ok", "clause_a", "note")
             if c in _pooled.columns]
    print(f"Human-match (gate clause A), gold TEST only -- gate on the CI LOWER "
          f"BOUND >= {CLAUSE_A_MIN:.2f} AND n_pos >= {CLAUSE_A_MIN_POS}, with the "
          f"ensemble counted present above {ENS_PRESENT:.2f}:")
    # na_rep, not a fallback inside float_format: pandas renders a null through
    # na_rep and never hands it to float_format, so the else branch of such a
    # lambda is dead code and the table prints the word NaN. An uncomputable
    # quantity shows as -- in every table in this notebook, and na_rep is what
    # makes that true rather than merely stated.
    print(_pooled[_show].to_string(
        index=False, float_format=lambda x: f"{x:.3f}", na_rep="--"))
    _plumb = [r["facet"] for r in hm_pooled if r["note"] in PLUMBING_NOTES]
    if len(_plumb) == len(FACET_NAMES):
        print()
        print("  PLUMBING FAULT, NOT A RESULT. Every facet reports its human")
        print("  side missing, which means the join or HUMAN_COL is wrong -- it")
        print("  does NOT mean nine facets failed against the coders. Check that")
        print("  the coder files use the plain facet names, or point HUMAN_COL at")
        print("  the suffix they use, and run DO (15) again. Do not read the table")
        print("  above as a gate result.")
    elif _plumb:
        print()
        print(f"  HUMAN SIDE MISSING for {_plumb}: those rows are a join")
        print("  problem, not a facet that failed. See the note column.")
    _thin = [r["facet"] for r in hm_pooled if r.get("boot_ok", 0) not in (0, 2000)]
    if _thin:
        print(f"  DEGENERATE BOOTSTRAP DRAWS on {_thin}: fewer than 2,000 of the")
        print("  resamples produced a number, so those intervals rest on the draws")
        print("  that caught more positive cases and are narrower than the full")
        print("  interval would be. Read them with the boot_ok column beside them.")
else:
    # Every facet gets a row saying so. A blank row, or a NaN, could be read later
    # as a facet that was tested against humans and failed; "not run" cannot.
    hm_pooled = [{"scope": "pooled", "facet": f, "clause_a": "not run",
                  "note": NOT_RUN_REASON} for f in FACET_NAMES]
    print("Clause A rows written as 'not run' for all nine facets.")

In [ ]:
# === DO (15 of 17, cont.): clause A BY STRATUM, never pooled into one number ===
# A single pooled agreement number is dominated by the easy cases and hides failure
# exactly where the desert-versus-revenge line is drawn. "He deserves it" is
# unanimous; the mixed-motive passage where a writer reaches for both desert and
# payback in one sentence is the case this study exists to read, and it is a small
# share of the sample. A facet that matches humans only on the clean stratum has
# not earned a primary claim (design section 4).
#
# Notebook 03 draws the sample in three strata and the names ride through into
# gold_codes.csv: 'random_stratified' (the stratified but unfiltered slice),
# 'survey_disagreement' -- or 'dict_vengeful' where no survey file was available --
# (the targeted draw: participants whose closed-ended answers and whose text
# disagree), and 'open_readthrough' (a fully random reserve). Clause A is reported
# on two CLASSES of stratum, because what matters is whether a facet matches humans
# where the construct boundary is actually contested:
#   boundary  the targeted stratum, where mixed-motive readings and shared blind
#             spots live -- the cases the desert-versus-revenge question turns on
#   clean     the two untargeted draws, i.e. the corpus as it comes
# Any stratum name added later counts as clean unless it is listed here, so a new
# stratum can never quietly join the boundary class and dilute it.
BOUNDARY_STRATA = {"survey_disagreement", "dict_vengeful"}

def _stratum_class(s):
    return "boundary" if str(s) in BOUNDARY_STRATA else "clean"

# Emptied here for the same reason hm_pooled is: this cell has to be re-runnable
# on its own without writing a second copy of every stratum row.
hm_stratum = []
if CLAUSE_A_RUNS and "stratum" in gold_test.columns:
    for f in FACET_NAMES:
        hc = HUMAN_COL(f)
        if human_side_missing(f):    # a join problem, already named in DO (15)
            continue
        for strat, sub in gold_test.groupby("stratum"):
            est, lo, hi, n_ok = bootstrap_agreement(sub[f"{f}_ens_mean"], sub[hc])
            hm_stratum.append({
                "scope": "stratum", "stratum": strat,
                "stratum_class": _stratum_class(strat), "facet": f,
                "est": est, "ci_lo": lo, "ci_hi": hi, "boot_ok": n_ok,
                "n_pos": int((pd.to_numeric(sub[hc], errors="coerce") > 0).sum()),
                "n_rows": int(len(sub)), "ens_present_at": ENS_PRESENT,
                "clause_a": "reported, not gated",
                "note": "per-stratum estimate; the verdict is the pooled row"})
    print(f"Per-stratum human-match computed (boundary strata: "
          f"{sorted(BOUNDARY_STRATA)}).")
elif CLAUSE_A_RUNS:
    print("No 'stratum' column in the consensus codes -- the by-stratum table is")
    print("skipped. A missing diagnostic should be visible in the run log, so this")
    print("line is printed rather than the table quietly omitted.")

In [ ]:
# === DO (15 of 17, cont.): the same comparison collapsed to two classes ===
# POOLED rather than averaged: the mean of two agreement coefficients computed on
# different numbers of rows is not an agreement coefficient. Only the point
# estimate is taken here -- these two numbers are read beside the verdict, never as
# the verdict, which is decided on the pooled interval above -- so the
# 2,000-resample bootstrap is not spent on them.
#
# WHERE THESE TWO NUMBERS GO. Notebook 10 reads the scope == 'stratum_class' rows
# of human_match_results.csv and puts their `est` on the gate table as
# human_match_clean and human_match_boundary, beside the clause A verdict they
# qualify. That is the whole destination: nothing else reads them.
#
# Emptied here, for the re-run reason given in the two cells above.
hm_class_rows = []
if CLAUSE_A_RUNS and "stratum" in gold_test.columns:
    gold_test["stratum_class"] = gold_test["stratum"].map(_stratum_class)
    for f in FACET_NAMES:
        hc = HUMAN_COL(f)
        if human_side_missing(f):    # a join problem, already named in DO (15)
            continue
        for cls in ("clean", "boundary"):
            sub = gold_test[gold_test["stratum_class"] == cls]
            hm_class_rows.append({
                "scope": "stratum_class", "stratum_class": cls, "facet": f,
                "est": _agree_point(sub[f"{f}_ens_mean"], sub[hc]),
                "n_pos": int((pd.to_numeric(sub[hc], errors="coerce") > 0).sum()),
                "n_rows": int(len(sub)), "ens_present_at": ENS_PRESENT,
                "clause_a": "reported, not gated",
                "note": "clean vs boundary point estimate"})
    _cls = pd.DataFrame(hm_class_rows)
    if len(_cls):
        print("Clean vs boundary human-match (point estimates):")
        print(_cls.pivot(index="facet", columns="stratum_class",
                         values="est").to_string(
            float_format=lambda x: f"{x:.3f}", na_rep="--"))
        print("  A facet strong on 'clean' and weak on 'boundary' has matched the")
        print("  coders on the easy cases only. That is not a gate-backed facet.")

In [ ]:
# === DO (16 of 17): shared-error diagnostic -- what averaging hides ===
# THE PROBLEM IT DETECTS, WHICH CLAUSE A CANNOT. Averaging three raters cancels
# INDEPENDENT error. It does not touch SHARED error -- it launders it. Three models
# that all read "pay for what he did" as pure desert when many writers mean payback
# produce three matching numbers, a low between-model variance, a high ICC, and a
# tight ensemble mean that is confidently wrong. Every reliability statistic in the
# chain reads that as a well-measured facet. The two quantities below are what
# distinguish a well-measured facet from a jointly mistaken one, and this is
# precisely the desert-versus-revenge line prereg 6.7 says the study exists to
# examine.
#
# THE MEAN IS NOT TRUSTED AS THE SCORE OF RECORD UNTIL THIS PASSES. Clause A shows
# the ensemble matches humans on average; this shows the residual error is roughly
# zero-mean and not correlated across raters, which is what has to hold for "mean
# across models" to be an estimator rather than a consensus opinion. A flagged
# facet is reported as biased and its gate-backed use is capped even where clause A
# and the ICC both pass (design section 4, and rule 7 of section 9).
#
# WHERE THE CAP IS APPLIED: not here. The flag travels on the pooled rows of
# human_match_results.csv, notebook 10 reads it in DO (7) and writes
# clauseB_capped_by_shared_error onto the gate table. Nothing is capped on a run
# with HUMAN_GOLD False, because there is no flag for notebook 10 to read.
shared_rows = []
if CLAUSE_A_RUNS:
    for f in FACET_NAMES:
        hc = HUMAN_COL(f)
        if human_side_missing(f):    # a join problem, already named in DO (15)
            continue
        hum = pd.to_numeric(gold_test[hc], errors="coerce").astype(float)
        ens = gold_test[f"{f}_ens_mean"].astype(float)
        # (1) SIGNED bias, deliberately not absolute. The sign IS the finding: a
        # positive value means the ensemble reads MORE of this facet into the
        # corpus than trained coders do, a negative value means it misses it. An
        # absolute-error summary would report the same magnitude for "reads revenge
        # into everything" and "cannot see revenge at all", which call for opposite
        # readings of every downstream E4/E7 result. Any agreement measure
        # forgiving of a constant offset would score a uniformly inflated facet
        # well; this is the check that catches it.
        signed_bias = float((ens - hum).mean(skipna=True))
        # (2) Each model's residual against the SAME human consensus, one error
        # series per rater, aligned row by row.
        err = {m: (gold_test[f"{f}__{m}"].astype(float) - hum)
               for m in MODELS if f"{f}__{m}" in gold_test.columns}
        shared_rows.append({"facet": f, "signed_bias": signed_bias,
                            "_err": pd.DataFrame(err)})

In [ ]:
# === DO (16 of 17, cont.): the cross-model error correlation, and the flags ===
# Mean pairwise correlation of those residuals. This is the shared-blind-spot
# signature. Near zero means the models are wrong in unrelated ways, which is the
# condition under which averaging genuinely cancels error and the mean is a better
# estimate than any single rater. Strongly positive means they are wrong on the
# SAME responses in the SAME direction -- the average is then no more accurate than
# one model, only more confident-looking, and the ICC is measuring a shared reading
# rather than a recoverable construct.
#
# The two thresholds that operationalise "roughly zero-mean and cross-model
# independent". Either one alone flags the facet:
#   |signed_bias| > 0.10   a systematic offset above a tenth of the 0-1 facet scale
#                          is large relative to the differences E3-E7 test, so it
#                          is not absorbable as noise
#   xcorr > 0.50           a Pearson correlation, so .50 means about a QUARTER of
#                          the variance in one model's errors is shared with
#                          another's, not half. Above it, averaging has stopped
#                          cancelling error and started concentrating it.
# A flagged facet is NOT dropped. It is reported with its numbers, notebook 10
# carries the flag onto the gate table, and clause B is capped there however high
# the ICC (design section 4, and rule 7 of section 9).
BIAS_MAX, XCORR_MAX = 0.10, 0.50
shared_error = {}
for rec in shared_rows:
    # READ, never removed. Popping "_err" out of rec would empty the dicts that
    # shared_rows still holds, so this cell would raise a KeyError the second time
    # it was run -- and this is the cell carrying BIAS_MAX and XCORR_MAX, which is
    # the one a reader is most likely to edit and run again.
    err_df = rec["_err"]
    cc = err_df.corr()
    cols = list(err_df.columns)
    pairs = [(a, b) for i, a in enumerate(cols) for b in cols[i + 1:]]
    # The non-NaN pairwise correlations only, and NaN where there are none. Asking
    # numpy for the mean of an empty selection is a question with no answer, and it
    # answers with NaN and a warning rather than refusing; deciding it here means
    # the NaN in the table is this notebook's statement and not numpy's shrug.
    _cx = [cc.loc[a, b] for a, b in pairs]
    _cx = [v for v in _cx if v == v]
    xcorr = float(np.mean(_cx)) if _cx else np.nan
    biased = bool(abs(rec["signed_bias"]) > BIAS_MAX
                  or (not np.isnan(xcorr) and xcorr > XCORR_MAX))
    # A fresh dict per facet, so nothing here mutates what cell DO (16) built.
    shared_error[rec["facet"]] = {
        "facet": rec["facet"], "signed_bias": rec["signed_bias"],
        "cross_model_err_corr": xcorr,
        "shared_error_flag": "biased" if biased else "ok"}

if shared_error:
    # WHAT THIS TABLE MEANS. 'ok' on a facet means: the ensemble is not
    # systematically higher or lower than the coders, and where it is wrong the
    # models are wrong independently -- so the mean across models can be trusted as
    # that facet's score of record. 'biased' means the ensemble mean for that facet
    # is a shared reading, and no ICC, however high, repairs it.
    print("Shared-error diagnostic (gold TEST): signed bias and cross-model error "
          "correlation.")
    print(pd.DataFrame(shared_error.values()).to_string(
        index=False, float_format=lambda x: f"{x:.3f}", na_rep="--"))
else:
    print("Shared-error diagnostic: not run.")

In [ ]:
# === DO (16 of 17, cont.): the open-weight model on its own ===
# Reported on its own line, never folded into the consensus. The open-weight leg is
# the architecturally most different rater in the panel -- different pretraining
# mix, different post-training, different lab -- so when it dissents from the
# proprietary models that dissent is evidence about a possible shared blind spot in
# the majority, not an outlier to be voted down (design section 4).
#
# The majority-vote failure mode this guards against: two similar models agree, the
# third is overruled, the mean and the ICC both look excellent, and the one rater
# with any chance of seeing the blind spot has been averaged away. Reading this
# beside the cross-model error correlation above is what makes that visible -- an
# open-weight standalone match well below the ensemble's, on a facet whose errors
# correlate across the proprietary pair, is the signature.
open_standalone = {}
if CLAUSE_A_RUNS and MODELS:
    open_key = "open" if "open" in MODELS else list(MODELS)[-1]
    for f in FACET_NAMES:
        hc = HUMAN_COL(f)
        # The human side has to be there, and so does this model's own column.
        # "no ensemble column" is not a reason to skip here: that one is about the
        # ENSEMBLE MEAN, and this comparison does not use the mean at all.
        if (human_side_missing(f) in ("no human column", "human column is empty")
                or f"{f}__{open_key}" not in gold_test.columns):
            continue
        est, lo, hi, _n_ok = bootstrap_agreement(gold_test[f"{f}__{open_key}"],
                                                 gold_test[hc])
        open_standalone[f] = {"open_model": open_key, "open_model_match_est": est,
                              "open_ci_lo": lo, "open_ci_hi": hi}
    if open_standalone:
        print(f'Open-weight model ("{open_key}") STANDALONE human-match '
              "(reported separately, never outvoted):")
        print(pd.DataFrame(open_standalone.values(),
                           index=list(open_standalone.keys())).to_string(
            float_format=lambda x: f"{x:.3f}", na_rep="--"))
else:
    print("Open-weight standalone match: not run.")

In [ ]:
# === DO (17 of 17): assemble the one results table ===
# One file, one row per (scope, facet). The scope column says what a row is:
#   pooled          the clause A verdict for that facet, on the whole TEST half
#   stratum         the same estimate within one stratum, reported not gated
#   stratum_class   the clean/boundary collapse, reported not gated
# The shared-error and open-weight columns attach to the pooled rows, because they
# are read together with the verdict and separating them into another file makes
# it possible to read one without the other.
human_match_df = pd.DataFrame(hm_pooled + hm_stratum + hm_class_rows)
# Any column a run did not produce is created empty so the file always has the
# same shape. WORD COLUMNS ARE CREATED AS object, NUMBER COLUMNS AS float: a
# column created from np.nan alone is a float column, and writing the word
# 'biased' into it below is a silent dtype coercion that pandas has announced it
# will stop allowing. Creating each column as what it will hold keeps the file
# the same on every pandas version.
for col in ("est", "ci_lo", "ci_hi", "ci_width", "n_pos", "n_rows", "boot_ok",
            "ens_present_at", "signed_bias", "cross_model_err_corr",
            "open_model_match_est", "open_ci_lo", "open_ci_hi"):
    if col not in human_match_df.columns:
        human_match_df[col] = np.full(len(human_match_df), np.nan)
for col in ("scope", "stratum", "stratum_class", "shared_error_flag",
            "open_model"):
    if col not in human_match_df.columns:
        human_match_df[col] = pd.Series(np.nan, index=human_match_df.index,
                                        dtype="object")
    else:
        human_match_df[col] = human_match_df[col].astype("object")

_pooled_mask = human_match_df["scope"] == "pooled"
for i, r in human_match_df[_pooled_mask].iterrows():
    se = shared_error.get(r["facet"], {})
    for k in ("signed_bias", "cross_model_err_corr", "shared_error_flag"):
        if k in se:
            human_match_df.at[i, k] = se[k]
    os_ = open_standalone.get(r["facet"], {})
    for k, v in os_.items():
        human_match_df.at[i, k] = v

# THE CONTRACT NOTEBOOK 10 READS, CHECKED BEFORE THE FILE IS BUILT. Notebook 10
# filters this file to scope == 'pooled' and takes one verdict per facet; where it
# finds two pooled rows for a facet it records clause A as "not run" for EVERY
# facet rather than let whichever row sorted last decide. So a duplicate here does
# not corrupt the gate -- it silently empties clause A, and the gate table then
# looks correctly computed. That is why it stops the notebook instead.
_p = human_match_df[human_match_df["scope"] == "pooled"]
assert not _p.duplicated(["facet"]).any(), (
    "human_match_results.csv would carry more than one pooled row for "
    f"{sorted(_p[_p.duplicated(['facet'], keep=False)]['facet'].unique())}. "
    "Notebook 10 reads clause A as 'not run' when that happens. Run DO (15) "
    "through DO (17) in order, from the top of DO (15).")
assert len(_p) == len(FACET_NAMES), (
    f"human_match_results.csv would carry {len(_p)} pooled rows for "
    f"{len(FACET_NAMES)} facets. Every facet needs exactly one, carrying PASS, "
    "FAIL or 'not run' -- a facet with no row at all reads downstream as a facet "
    "nobody has a verdict for. Run DO (15) through DO (17) in order.")

human_match_df = human_match_df[[
    "scope", "stratum", "stratum_class", "facet", "est", "ci_lo", "ci_hi",
    "ci_width", "n_pos", "n_rows", "boot_ok", "ens_present_at", "clause_a",
    "note", "signed_bias", "cross_model_err_corr", "shared_error_flag",
    "open_model", "open_model_match_est", "open_ci_lo", "open_ci_hi"]]
print(f"Results table assembled: {len(human_match_df)} rows "
      f"({int(_pooled_mask.sum())} of them the pooled clause A verdicts, one per "
      "facet).")

## Save and check

In [ ]:
# === SAVE: every output through done() ===
# codebook_bottom_up.txt is registered even though it holds no data: it is coder
# material, it is part of what the OSF amendment carries, and a later reader has to
# be able to see which version of the definitions a coding pass used.
done(CODEBOOK_OUT, rows=None,
     note="bottom-up coder materials: definitions only, no anchors, no pinned tokens")

# gold_codes.csv is written only where there was something to adjudicate. An empty
# file here would be worse than no file: the reference-pool notebook reads this one
# when HUMAN_GOLD is True, and a file of blank codes would quietly build reference
# passages out of nothing.
if gold_codes is not None:
    # ONE ROW PER response_id x item_type, CHECKED ON THE FRAME BEING WRITTEN.
    # Notebooks 04 and 11 both read this file and merge on that pair. A doubled row
    # would be embedded twice in a facet centroid in 04 and would count one
    # participant's sentence twice in E4 in 11, and neither would raise: the
    # printed sample size would just be wrong. The chain that should make this
    # true runs through DO (6), which sets aside any coder file with a repeated
    # row, and DO (6, cont.), whose intersection of unique indexes is unique. This
    # assertion is what catches a break anywhere along it.
    assert not gold_codes.duplicated(KEY).any(), (
        "gold_codes.csv would carry more than one row for "
        + str(gold_codes[gold_codes.duplicated(KEY, keep=False)][KEY]
              .head(4).to_dict("records"))
        + ". Nothing downstream would notice. Check the coder files for repeated "
        "rows and re-run from DO (6).")
    gold_codes.to_csv("gold_codes.csv", index=False)
    done("gold_codes.csv", rows=len(gold_codes),
         note=f"adjudicated consensus, {len(cids)} coders, TIE_RULE='{TIE_RULE}'")
else:
    print("  [not written] gold_codes.csv -- no coder files to adjudicate.")

if len(irr_df):
    irr_df.to_csv("gold_irr_by_facet.csv", index=False)
    done("gold_irr_by_facet.csv", rows=len(irr_df),
         note=f"human-side reliability: Gwet AC1 (primary), Cohen kappa "
              f"(reference), holistic alpha={alpha_hol}")
else:
    print("  [not written] gold_irr_by_facet.csv -- no reliability was computed.")

# This one is ALWAYS written. Where clause A did not run, it is written with
# clause_a = 'not run' on every facet, which is a different statement from a blank
# cell and from a failed facet, and the whole point of the file is to keep those
# three apart.
human_match_df.to_csv("human_match_results.csv", index=False)
done("human_match_results.csv", rows=len(human_match_df),
     note=f"gate clause A ({CLAUSE_A_STATUS}); pooled + per stratum + "
          f"clean/boundary; ensemble presence threshold {ENS_PRESENT}")

In [ ]:
# === CHECK: what was produced, and what to do next ===
print("NOTEBOOK 99 -- what was produced")
print(f"  coder files read:      {len(coder_files)}")
print(f"  usable raters:         {len(cids)}")
print(f"  consensus rows:        {0 if gold_codes is None else len(gold_codes)}")
print(f"  facets with an IRR:    {len(irr_df)}")
print(f"  clause A:              {CLAUSE_A_STATUS.upper()}"
      + (f"  ({NOT_RUN_REASON})" if CLAUSE_A_STATUS == "not run" else ""))
if CLAUSE_A_RUNS:
    _p = human_match_df[human_match_df["scope"] == "pooled"]
    print(f"    PASS: {sorted(_p[_p.clause_a == 'PASS'].facet)}")
    print(f"    FAIL: {sorted(_p[_p.clause_a == 'FAIL'].facet)}")
    print("  A PASS here makes a facet ELIGIBLE to be gate-backed. It is not the")
    print("  gate: clauses B and C still have to pass, and every E3-E7 claim stays")
    print("  exploratory whatever the gate says.")

if RUN_MODE == "smoke":
    print("\n  SMOKE RUN: these files describe a handful of participants, not the")
    print("  corpus. Nothing here may be posted or reported.")

print("\nWHO READS THIS, AND ONLY WHEN HUMAN_GOLD IS True")
print("  notebook 04   gold_codes.csv, for the human-coded reference passages")
print("  notebook 10   human_match_results.csv, for gate clause A and for the")
print("                shared-error flag that caps clause B")
print("  notebook 11   gold_codes.csv, as a non-model corroborator for E4")
print("  gold_irr_by_facet.csv is read by NO notebook. It goes on the OSF")
print("  amendment, and 'hard for humans, therefore descriptive-only' has to be")
print("  carried into the write-up by hand.")
if not HUMAN_GOLD:
    print("  HUMAN_GOLD IS FALSE ON THIS RUN, so none of those three will open")
    print("  anything written above, and _manifest.json records human_gold=False.")

print("\nGET THE FILES OFF THIS MACHINE BEFORE THE SESSION ENDS")
print(f"  Everything above was written to {WORKDIR}. A Colab machine's filesystem")
print("  is discarded when the runtime disconnects and there is no drive.mount")
print("  anywhere in this pipeline, so anything not copied out is gone -- and a")
print("  human coding pass is the one thing in this chain that cannot be re-run")
print("  cheaply. Download these, or copy them to Drive:")
# Listed by what is actually on disk, not by what a full run would produce: a
# reader told to rescue a file that was never written stops trusting the list.
for _f in [CODEBOOK_OUT, "gold_codes.csv", "gold_irr_by_facet.csv",
           "human_match_results.csv"]:
    if os.path.exists(os.path.join(WORKDIR, _f)):
        print(f"    {_f}")

print("\nWHAT THIS ARM IS, ONE LAST TIME")
print("  Prereg 6.7 prescribes no human-coding step. This whole notebook is a")
print("  DEVIATION and needs its dated OSF amendment posted BEFORE the coding")
print("  happened -- not after, and not at write-up.")
print("  The registered gate is unaffected either way: clause B (inter-model ICC")
print("  >= .60) and clause C (r >= .30 with the matching survey scale) run on the")
print("  default chain with no human anywhere in them.")